# Data analysis + split for CLIP classification (per-image, leakage-checked)

Reads `output.json` (image-caption pairs). First **analyses** the class distribution, then **drops ultra-rare
classes** that cannot be evaluated under 5-fold CV, then makes clean reproducible splits.

**Why drop rare classes:** under 5-fold CV a class needs **n>=5** to appear in *all* 5 test folds (1 per fold).
Classes with n<5 appear in only `min(n,5)` folds, so their per-class metric is missing from some folds and the
mean+/-std is unreliable. Set `MIN_SAMPLES` (default 5). Run Cell 2-3 first to see the trade-off, then choose.

Outputs (per-image, no grouping, stratified by caption, fixed seed):
- `splits_clip/classes.csv` - kept-class list (re-indexed labels)
- `splits_clip/cv5/fold0..4/{train,val,test}.csv` - 5-fold CV (~70/10/20); every image tested exactly once
- `splits_clip/single_811/{train,val,test}.csv` - single 8:1:1 dev split
- `splits_clip/class_distribution.png`


In [1]:
# Cell 1. Config + load
import json, warnings, collections
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

OUTPUT_JSON=Path("output.json")     # run from the folder with output.json + flickr8k-images/
OUT=Path("splits_clip"); OUT.mkdir(exist_ok=True)
SEED=42; N_FOLDS=5; VAL_FOLDS=8
SINGLE_TEST=0.10; SINGLE_VAL=0.10
MIN_SAMPLES=5                       # <-- drop classes with fewer than this many samples (see Cell 3 trade-off)

assert OUTPUT_JSON.exists(), f"output.json not found in {Path.cwd()}"
data=json.load(open(OUTPUT_JSON,encoding="utf-8"))
def cap(d):
    c=d["caption"]; return (c[0] if isinstance(c,list) else c)
df_all=pd.DataFrame([{"image":d["image"],"caption_raw":str(cap(d)),"caption_norm":str(cap(d)).strip().lower()} for d in data])
print(f"loaded {len(df_all)} samples, {df_all['caption_norm'].nunique()} unique classes")


loaded 656 samples, 46 unique classes


In [2]:
# Cell 2. ANALYSIS - class distribution
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
vc=df_all["caption_norm"].value_counts()
N=len(df_all)
print(f"samples={N}  classes={len(vc)}")
print(f"class size: min={vc.min()} max={vc.max()} median={int(vc.median())} mean={vc.mean():.1f}")
print()
print("=== how many classes fall at each sample count ===")
sh=collections.Counter(vc.values)
for s in sorted(sh): print(f"  n={s:>2}: {sh[s]} class(es)")
print()
print("=== full per-class counts (sorted) ===")
for c,n in vc.items(): print(f"  {n:>3}  {c}")
# bar chart (useful as a paper figure)
fig,ax=plt.subplots(figsize=(10,7))
ax.barh(range(len(vc)), vc.values[::-1]); ax.set_yticks(range(len(vc)))
ax.set_yticklabels(vc.index[::-1], fontsize=6); ax.set_xlabel("number of images")
ax.axvline(MIN_SAMPLES-0.5, color="r", ls="--", lw=1, label=f"MIN_SAMPLES={MIN_SAMPLES}")
ax.set_title("Per-class sample counts"); ax.legend(); plt.tight_layout()
plt.savefig(OUT/"class_distribution.png", dpi=160, bbox_inches="tight"); plt.show()
print("\nsaved", OUT/"class_distribution.png")


samples=656  classes=46
class size: min=2 max=94 median=5 mean=14.3

=== how many classes fall at each sample count ===
  n= 2: 8 class(es)
  n= 3: 5 class(es)
  n= 4: 3 class(es)
  n= 5: 7 class(es)
  n= 6: 4 class(es)
  n= 7: 1 class(es)
  n= 9: 1 class(es)
  n=11: 1 class(es)
  n=12: 3 class(es)
  n=13: 1 class(es)
  n=14: 1 class(es)
  n=22: 4 class(es)
  n=33: 1 class(es)
  n=35: 2 class(es)
  n=47: 1 class(es)
  n=52: 1 class(es)
  n=80: 1 class(es)
  n=94: 1 class(es)

=== full per-class counts (sorted) ===
   94  hole and pitting in deck plate
   80  hole in deck at soffit
   52  section loss in bowstring arch n-type truss flange angles
   47  hole in deck plate
   35  section loss in crank on main girder external face
   35  section loss in lower crank angle
   33  section loss in crank on main girder internal face
   22  section loss in flange on cross girder
   22  upper rolled steel angle of bottom boom corroded on truss outside elevation
   22  corroded drainage spigot at 

In [3]:
# Cell 3. TRADE-OFF table for the drop threshold (5-fold needs n>=5 for full coverage)
print("A class needs n>=%d to appear in ALL %d test folds (1 per fold). n<%d -> appears in only min(n,%d) folds.\n"%(N_FOLDS,N_FOLDS,N_FOLDS,N_FOLDS))
print(f"{'threshold n>=t':<16}{'keep cls':>9}{'drop cls':>9}{'keep smp':>9}{'drop smp':>9}{'drop %':>8}{'min fold-cov':>14}")
for t in [2,3,4,5,6,8,10]:
    keep=vc[vc>=t]; drop=vc[vc<t]
    ks=int(keep.sum()); ds=N-ks; mincov=min(min(n,N_FOLDS) for n in keep.values)
    print(f"  n>={t:<12}{len(keep):>9}{len(drop):>9}{ks:>9}{ds:>9}{100*ds/N:>7.1f}%{mincov:>11}/{N_FOLDS}")
print(f"\nCurrent MIN_SAMPLES={MIN_SAMPLES}.  Classes that WILL be dropped:")
for c,n in vc[vc<MIN_SAMPLES].sort_values().items(): print(f"  n={n}  {c}")
print(f"\n-> keeping {int((vc>=MIN_SAMPLES).sum())} classes / {int(vc[vc>=MIN_SAMPLES].sum())} images; "
      f"dropping {int((vc<MIN_SAMPLES).sum())} classes / {N-int(vc[vc>=MIN_SAMPLES].sum())} images "
      f"({100*(N-int(vc[vc>=MIN_SAMPLES].sum()))/N:.1f}%).")


A class needs n>=5 to appear in ALL 5 test folds (1 per fold). n<5 -> appears in only min(n,5) folds.

threshold n>=t   keep cls drop cls keep smp drop smp  drop %  min fold-cov
  n>=2                  46        0      656        0    0.0%          2/5
  n>=3                  38        8      640       16    2.4%          3/5
  n>=4                  33       13      625       31    4.7%          4/5
  n>=5                  30       16      613       43    6.6%          5/5
  n>=6                  23       23      578       78   11.9%          5/5
  n>=8                  18       28      547      109   16.6%          5/5
  n>=10                 17       29      538      118   18.0%          5/5

Current MIN_SAMPLES=5.  Classes that WILL be dropped:
  n=2  hole throughout caisson
  n=2  hole in web at rail bearer
  n=2  section loss in splice plate on truss outside elevation
  n=2  hole in web of truss and full section loss of upper rolled steel angle
  n=2  corroded end plate on truss o

In [4]:
# Cell 4. Apply MIN_SAMPLES filter -> kept df + re-indexed class list
keep_classes=sorted(vc[vc>=MIN_SAMPLES].index)
df=df_all[df_all["caption_norm"].isin(keep_classes)].reset_index(drop=True).copy()
CLASSES=sorted(df["caption_norm"].unique()); CIDX={c:i for i,c in enumerate(CLASSES)}
df["label"]=df["caption_norm"].map(CIDX)
pd.DataFrame({"label":range(len(CLASSES)),"caption":CLASSES}).to_csv(OUT/"classes.csv",index=False,encoding="utf-8-sig")
print(f"after filter: {len(df)} samples, {len(CLASSES)} classes")
print("saved", OUT/"classes.csv")


after filter: 613 samples, 30 classes
saved splits_clip/classes.csv


In [5]:
# Cell 5. 5-fold stratified CV (per-image) + val carved, leakage-safe
y=df["caption_norm"].values; N=len(df); idx=np.arange(N)
cv_dir=OUT/f"cv{N_FOLDS}"; cv_dir.mkdir(exist_ok=True)
skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
summary=[]; all_test=[]
for k,(tv,te) in enumerate(skf.split(idx,y)):
    cnt=collections.Counter(y[tv])
    elig=np.array([i for i in tv if cnt[y[i]]>=2])
    inner=StratifiedKFold(n_splits=VAL_FOLDS,shuffle=True,random_state=SEED)
    _,vsel=next(inner.split(elig,y[elig])); va=elig[vsel]
    tr=np.array(sorted(set(tv.tolist())-set(va.tolist())))
    assert not(set(tr)&set(va)) and not(set(tr)&set(te)) and not(set(va)&set(te)), "overlap!"
    assert set(y[te])<=set(y[tr]) and set(y[va])<=set(y[tr]), "a val/test class missing from train!"
    d=cv_dir/f"fold{k}"; d.mkdir(exist_ok=True)
    for nm,ii in [("train",tr),("val",va),("test",te)]:
        df.iloc[ii].to_csv(d/f"{nm}.csv",index=False,encoding="utf-8-sig")
    all_test+=te.tolist(); summary.append({"fold":k,"train":len(tr),"val":len(va),"test":len(te)})
print(pd.DataFrame(summary).to_string(index=False))
assert sorted(all_test)==list(range(N)), "not every image tested exactly once!"
print(f"\nOK: every image tested exactly once across {N_FOLDS} folds. wrote -> {cv_dir}")


 fold  train  val  test
    0    428   62   123
    1    428   62   123
    2    428   62   123
    3    429   62   122
    4    429   62   122

OK: every image tested exactly once across 5 folds. wrote -> splits_clip/cv5


In [6]:
# Cell 6. Single 8:1:1 split (per-class; guarantees >=1 train per class)
rng=np.random.RandomState(SEED); tr_i,va_i,te_i=[],[],[]
for c in CLASSES:
    ii=df.index[df["caption_norm"]==c].to_numpy().copy(); rng.shuffle(ii); n=len(ii)
    nte=int(round(SINGLE_TEST*n)); nva=int(round(SINGLE_VAL*n))
    while n-nte-nva<1 and (nte>0 or nva>0):
        if nte>=nva and nte>0: nte-=1
        elif nva>0: nva-=1
    te_i+=ii[:nte].tolist(); va_i+=ii[nte:nte+nva].tolist(); tr_i+=ii[nte+nva:].tolist()
assert not(set(tr_i)&set(va_i)) and not(set(tr_i)&set(te_i)) and not(set(va_i)&set(te_i))
assert set(df.loc[te_i,"caption_norm"])<=set(df.loc[tr_i,"caption_norm"])
sd=OUT/"single_811"; sd.mkdir(exist_ok=True)
for nm,ii in [("train",tr_i),("val",va_i),("test",te_i)]:
    df.iloc[ii].to_csv(sd/f"{nm}.csv",index=False,encoding="utf-8-sig")
print("single 8:1:1 -> train=%d val=%d test=%d | test covers %d/%d classes"%(
      len(tr_i),len(va_i),len(te_i),df.loc[te_i,"caption_norm"].nunique(),len(CLASSES)))
print("wrote ->", sd)


single 8:1:1 -> train=497 val=58 test=58 | test covers 23/30 classes
wrote -> splits_clip/single_811


In [7]:
# Cell 7. Verify + summary
print("=== files under", OUT.resolve(), "===")
for p in sorted(OUT.rglob("*.csv")): print("  ", p.relative_to(OUT), "(%d rows)"%len(pd.read_csv(p)))
cov=collections.Counter()
for k in range(N_FOLDS):
    for c in pd.read_csv(cv_dir/f"fold{k}"/"test.csv")["caption_norm"].unique(): cov[c]+=1
cs=pd.Series(cov)
print("\nper-class #folds-tested: min=%d max=%d"%(cs.min(),cs.max()),
      "(min should be %d if MIN_SAMPLES>=%d)"%(N_FOLDS,N_FOLDS))
print("classes never in any test fold:", [c for c in CLASSES if cov.get(c,0)==0] or "none (good)")
print("\nNEXT: point the CLIP screening notebook FOLD_DIR ->", (cv_dir/"fold0").as_posix())
print("      final run: loop fold0..fold%d, pool the 5 test predictions -> one confusion matrix + mean/std."%(N_FOLDS-1))


=== files under /root/autodl-tmp/CLIP_v4/splits_clip ===
   classes.csv (30 rows)
   cv5/fold0/test.csv (123 rows)
   cv5/fold0/train.csv (428 rows)
   cv5/fold0/val.csv (62 rows)
   cv5/fold1/test.csv (123 rows)
   cv5/fold1/train.csv (428 rows)
   cv5/fold1/val.csv (62 rows)
   cv5/fold2/test.csv (123 rows)
   cv5/fold2/train.csv (428 rows)
   cv5/fold2/val.csv (62 rows)
   cv5/fold3/test.csv (122 rows)
   cv5/fold3/train.csv (429 rows)
   cv5/fold3/val.csv (62 rows)
   cv5/fold4/test.csv (122 rows)
   cv5/fold4/train.csv (429 rows)
   cv5/fold4/val.csv (62 rows)
   single_811/test.csv (58 rows)
   single_811/train.csv (497 rows)
   single_811/val.csv (58 rows)

per-class #folds-tested: min=5 max=5 (min should be 5 if MIN_SAMPLES>=5)
classes never in any test fold: none (good)

NEXT: point the CLIP screening notebook FOLD_DIR -> splits_clip/cv5/fold0
      final run: loop fold0..fold4, pool the 5 test predictions -> one confusion matrix + mean/std.


## How to use
1. Put this notebook in the folder with `output.json` + `flickr8k-images/`.
2. Run **Cell 1-3 first** to see the distribution and the drop-threshold trade-off. Decide `MIN_SAMPLES`
   (default **5** = every kept class appears in all 5 folds; set 3 to keep more classes; 2 to keep all).
3. Re-run Cell 1 (with your chosen `MIN_SAMPLES`) then Cell 4-7 to write the splits.
4. Use `splits_clip/classes.csv` everywhere for consistent label indices.

Reporting guidance
- per-class P/R/F1 + confusion matrix + mean+/-std -> from the **5-fold pooled** result (every image tested once).
- In the paper, state plainly: classes with fewer than `MIN_SAMPLES` samples were excluded because they cannot
  be evaluated under 5-fold CV (directly answers the reviewer's small-sample concern).
- All splits are per-image (no grouping, your choice), stratified by caption, fixed seed -> reproducible.
